In [114]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches
from src.elo import prepare_matches, update_elo
from src.simulation import predict_match
from src.features import update_team_history, update_h2h
import numpy as np
from collections import Counter
from src.simulation import run_tournament


In [115]:
#Load everything from earlier notebooks

history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
features = joblib.load('model_features.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")

Singular run to demonstrate the process

In [116]:
#Group stage simulation

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0,
                     "GD": 0,
                     'GF': 0,
                     "GA": 0})
        
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
for match in group_stage_matches.itertuples(index = False):
    
    if match.home_team not in country_elo or pd.isna(country_elo[match.home_team]):
        country_elo[match.home_team] = 1500.0
    if match.away_team not in country_elo or pd.isna(country_elo[match.away_team]):
        country_elo[match.away_team] = 1500.0
    
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[match.home_team], country_elo[match.away_team])
    #print(f"{match.home_team} vs {match.away_team} ({x.home_score}-{x.away_score}, result: {x.outcome}), Prob: {x.probability}, diff: {x.diff}")
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GD'] += x.home_score - x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GD'] += x.away_score - x.home_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GF'] += x.home_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GF'] += x.away_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GA'] += x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GA'] += x.home_score
    
    if x.outcome == "Home Win": S = 1
    elif x.outcome == "Away Win": S = 0
    else: S = 0.5
    # new_away, new_home = update_elo(S, match.neutral, match.K_factor,x.home_score, x.away_score, 
    #                                 country_elo[match.away_team], 
    #                                 country_elo[match.home_team])
    
    # country_elo[match.home_team] = new_home
    # country_elo[match.away_team] = new_away
    # update_team_history(match, x.home_score, x.away_score, history_dict)
    # update_h2h(match, h2h_dict)
    
group_stage_result = group_stage_result.sort_values(['Group', 'Points', 'GD', 'GF'], ascending=[True, False, False, False]).reset_index(drop=True)    
print(group_stage_result) #Predicted group stage results




   Group                    Team  Points  GD  GF  GA
0      A                  Mexico       9   6   7   1
1      A             South Korea       4   0   8   8
2      A            South Africa       3  -2   8  10
3      A          Czech Republic       1  -4   5   9
4      B             Switzerland       9   4  10   6
5      B                  Canada       6   4   8   4
6      B  Bosnia and Herzegovina       1  -3   1   4
7      B                   Qatar       1  -5   4   9
8      C                 Morocco       7   2   4   2
9      C                  Brazil       6   3   8   5
10     C                Scotland       2  -1   5   6
11     C                   Haiti       1  -4   1   5
12     D                Paraguay       7   4   9   5
13     D                  Turkey       4   0   9   9
14     D           United States       2  -1   6   7
15     D               Australia       2  -3   3   6
16     E                 Ecuador       7   2   4   2
17     E             Ivory Coast       5   1  

In [117]:
#Get teams that move on to round of 32

top2 = group_stage_result.groupby('Group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('Group').nth(2).copy() #All teams that placed third (only 8 of them move on)

best8_third = third.sort_values(
    ['Points', 'GD', 'GF'], 
    ascending=[False, False, False]
).head(8)

thirds_slot_map = { #Needs to be fixed
    'E': 'ABCDEFGHIJKL',
    'I': 'ABCDEFGHIJKL', 
    'A': 'ABCDEFGHIJKL',
    'L': 'ABCDEFGHIJKL',
    'D': 'ABCDEFGHIJKL',
    'G': 'ABCDEFGHIJKL',
    'B': 'ABCDEFGHIJKL',
    'K': 'ABCDEFGHIJKL',
}

winners = {g: teams.iloc[0]['Team'] for g, teams in group_stage_result.groupby('Group')}
runners = {g: teams.iloc[1]['Team'] for g, teams in group_stage_result.groupby('Group')}

available_thirds = {row.Group: row.Team for row in best8_third.itertuples()}
assignments = {}
used = set()
for winner_group, eligible_groups in thirds_slot_map.items():
    for g in eligible_groups:
        if g in available_thirds and g not in used:
            assignments[winner_group] = available_thirds[g]
            used.add(g)
            break

r32_matchups = {
    73: (runners['A'], runners['B']),
    74: (winners['E'], assignments['E']),
    75: (winners['F'], runners['C']),
    76: (winners['C'], runners['F']),
    77: (winners['I'], assignments['I']),
    78: (runners['E'], runners['I']),
    79: (winners['A'], assignments['A']),
    80: (winners['L'], assignments['L']),
    81: (winners['D'], assignments['D']),
    82: (winners['G'], assignments['G']),
    83: (runners['K'], runners['L']),
    84: (winners['H'], runners['J']),
    85: (winners['B'], assignments['B']),
    86: (winners['J'], runners['H']),
    87: (winners['K'], assignments['K']),
    88: (runners['D'], runners['G'])
}

r32_rows = []
r16_teams = {}

for match, teams in r32_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])
    
    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    r16_teams[match] = winner
    r32_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    # new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
    #                                 country_elo[teams[0]], 
    #                                 country_elo[teams[1]])
    
    # country_elo[teams[0]] = new_home
    # country_elo[teams[1]] = new_away


South Korea 1 - 0 Canada → South Korea
Ecuador 1 - 0 South Africa → Ecuador
Sweden 8 - 9 Brazil → Brazil
Morocco 2 - 2 Tunisia → Tunisia
France 2 - 0 Germany → France
Ivory Coast 1 - 1 Norway → Norway
Mexico 5 - 6 Netherlands → Netherlands
England 4 - 2 New Zealand → England
Paraguay 1 - 1 Uruguay → Paraguay
Belgium 6 - 0 Iraq → Belgium
Portugal 3 - 1 Croatia → Portugal
Spain 2 - 0 Austria → Spain
Switzerland 4 - 0 Algeria → Switzerland
Argentina 1 - 1 Saudi Arabia → Argentina
Colombia 3 - 0 Uzbekistan → Colombia
Turkey 2 - 2 Egypt → Turkey


In [ ]:
#Round of 16
r16_matchups = {
    89: (r16_teams[74], r16_teams[77]),
    90: (r16_teams[73], r16_teams[75]),
    91: (r16_teams[76], r16_teams[78]),
    92: (r16_teams[79], r16_teams[80]),
    93: (r16_teams[83], r16_teams[84]),
    94: (r16_teams[81], r16_teams[82]),
    95: (r16_teams[86], r16_teams[88]),
    96: (r16_teams[85], r16_teams[87])
}

r16_rows = []
r8_teams = {}

for match, teams in r16_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    r8_teams[match] = winner
    r16_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    # new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
    #                                 country_elo[teams[0]], 
    #                                 country_elo[teams[1]])
    
    # country_elo[teams[0]] = new_home
    # country_elo[teams[1]] = new_away

print(country_elo)


Ecuador 1 - 1 France → Ecuador
South Korea 1 - 0 Brazil → South Korea
Tunisia 0 - 1 Norway → Norway
Netherlands 3 - 1 England → Netherlands
Portugal 1 - 1 Spain → Spain
Paraguay 2 - 3 Belgium → Belgium
Argentina 1 - 1 Turkey → Turkey
Switzerland 1 - 1 Colombia → Colombia


In [119]:
#Quarter final
QF_matchups = {
  97: (r8_teams[89], r8_teams[90]),
  98: (r8_teams[93], r8_teams[94]), 
  99: (r8_teams[91], r8_teams[92]), 
  100: (r8_teams[95], r8_teams[96]), 
}

QF_rows = []
SF_teams = {}

for match, teams in QF_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    SF_teams[match] = winner
    QF_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    # new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
    #                                 country_elo[teams[0]], 
    #                                 country_elo[teams[1]])
    
    # country_elo[teams[0]] = new_home
    # country_elo[teams[1]] = new_away


Ecuador 0 - 0 South Korea → South Korea
Spain 2 - 2 Belgium → Spain
Norway 1 - 1 Netherlands → Netherlands
Turkey 1 - 1 Colombia → Turkey


In [120]:
#Semi-Final

SF_matchups = {
  101: (SF_teams[97], SF_teams[98]),
  102: (SF_teams[99], SF_teams[100]) 
}

SF_rows = []
FINAL_teams = {}

for match, teams in SF_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    FINAL_teams[match] = winner
    SF_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    # new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
    #                                 country_elo[teams[0]], 
    #                                 country_elo[teams[1]])
    
    # country_elo[teams[0]] = new_home
    # country_elo[teams[1]] = new_away

South Korea 0 - 1 Spain → Spain
Netherlands 1 - 0 Turkey → Netherlands


In [121]:
for match, teams in {103: (FINAL_teams[101], FINAL_teams[102])}.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]
        
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

print(country_elo)

Spain 2 - 0 Netherlands → Spain
{'Spain': np.float64(2153.9129173895753), 'Argentina': np.float64(2113.4271620560976), 'France': np.float64(2062.1556144927195), 'England': np.float64(2019.6773854069866), 'Colombia': np.float64(1976.7364292760565), 'Brazil': np.float64(1989.655734793957), 'Portugal': np.float64(1984.4981931977347), 'Netherlands': np.float64(1944.192642161004), 'Ecuador': np.float64(1936.9654294259358), 'Croatia': np.float64(1909.363843471548), 'Norway': np.float64(1916.8147689670334), 'Germany': np.float64(1925.6371891713034), 'Switzerland': np.float64(1893.4797705606502), 'Uruguay': np.float64(1891.7243225970883), 'Turkey': np.float64(1905.3920054570926), 'Japan': np.float64(1905.9724729356008), 'Denmark': np.float64(1845.8105719807527), 'Italy': np.float64(1858.391607437261), 'Belgium': np.float64(1887.7675481912827), 'Mexico': np.float64(1875.6340963693294), 'Paraguay': np.float64(1835.5821055861488), 'Morocco': np.float64(1876.9553920741762), 'Austria': np.float64(1

In [122]:
results = []
for i in range(1000):
    w = run_tournament(wc_model, draw_threshold, history_dict, h2h_dict,
                       country_elo, model_h, model_a, scaler, group_stage_matches)
    results.append(w)

for team, count in Counter(results).most_common(10):
    print(f"{team}: {count/10:.1f}%")

KeyboardInterrupt: 